# FRIGID pilot (DEV run, NOT submission)

- Goal: 10-20 molecule feasibility (round0, ICEBERG OFF). Dual path: oracle-formula (diagnostic upper bound) vs MIST-CF Top1 predicted-formula (real path, E4 verified 15/20).
- Weights: Zenodo 19685145 `frigid_pretrained_checkpoints.tar.gz` (expect 2804076946 B, md5 1059c193b5f3dd7034079076e943389b), FRIGID repo coleygroup/FRIGID (Apache-2.0), paper arXiv:2604.16648.
- Contract isolation: MIST-4096 input only. FPNet-6930 / DreamsFP / DreaMS-1024 are NEVER imported here. Formula arms are labeled oracle vs mistcf-top1.
- B-track: all 20 pilot molecules are train-library-in; scored as library-out by excluding truth from pool (truth_in_pool flag + exclude-then-rank).
- Degrade plan: 20 -> 10 -> 5 -> 1 molecule. Any stuck step reports exact error, no fabrication. Internet ON is DEV-only.

In [ ]:
import sys, os, glob, json, time
print('python', sys.version)
print('INPUT_ROOT', sorted(os.listdir('/kaggle/input')))
hits = sorted(glob.glob('/kaggle/input/casmi26-frigid-pilot-20/**/*', recursive=True))
print('pilot inputs:', len(hits))
for p in hits[:10]: print(p)
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print('total_mem_GB', round(torch.cuda.get_device_properties(0).total_memory/1e9, 3))
import urllib.request
try:
    print('zenodo_api_status:', urllib.request.urlopen('https://zenodo.org/api/records/19685145', timeout=20).status)
except Exception as e:
    print('NET_FAIL:', repr(e))
import subprocess as _sp
print('--- /kaggle/input tree (top) ---')
_r = _sp.run(['find', '/kaggle/input', '-maxdepth', '3'], capture_output=True, text=True)
print((_r.stdout or '')[:2000])
_pj = sorted(glob.glob('/kaggle/input/**/pilot_20.json', recursive=True))
print('pilot_20.json candidates:', _pj)

In [ ]:
import time, hashlib, urllib.request, os
URL = 'https://zenodo.org/api/records/19685145/files/frigid_pretrained_checkpoints.tar.gz/content'
DST = '/kaggle/working/frigid_pretrained_checkpoints.tar.gz'
EXPECT_SIZE = 2804076946
EXPECT_MD5 = '1059c193b5f3dd7034079076e943389b'
# NOTE: sandbox once measured ~90KB/s; that is NOT worker bandwidth (workers are usually tens of MB/s).
# Judge ONLY by the worker log below: sustained <0.5MB/s with ETA>4h => stop kernel from web UI and report.
t0 = time.time(); n = 0; last_mark = 0; t_mark = t0
req = urllib.request.Request(URL, headers={'User-Agent': 'kaggle-frigid-pilot/2'})
with urllib.request.urlopen(req, timeout=60) as r, open(DST, 'wb') as f:
    while True:
        ch = r.read(1 << 20)
        if not ch: break
        f.write(ch); n += len(ch)
        if n - last_mark >= 256 * (1 << 20):
            dt = time.time() - t0
            inst = (n - last_mark) / max(time.time() - t_mark, 1e-6) / 1e6
            avg = n / 1e6 / max(dt, 1e-6)
            eta = (EXPECT_SIZE - n) / max(n / max(dt, 1e-6), 1e-9)
            print(f'downloaded {n/1e9:.2f}/{EXPECT_SIZE/1e9:.2f} GB | inst {inst:.1f} MB/s | avg {avg:.1f} MB/s | ETA {eta/3600:.2f}h', flush=True)
            last_mark = n; t_mark = time.time()
dt = time.time() - t0
print(f'download done: {n} B in {dt:.1f}s = {n/1e6/max(dt,1e-6):.1f} MB/s')
assert n == EXPECT_SIZE, f'SIZE_MISMATCH got {n} expect {EXPECT_SIZE}'
h = hashlib.md5()
with open(DST, 'rb') as f:
    for ch in iter(lambda: f.read(1 << 20), b''): h.update(ch)
print('md5', h.hexdigest(), 'match', h.hexdigest() == EXPECT_MD5)
assert h.hexdigest() == EXPECT_MD5, 'MD5_MISMATCH'
print('WEIGHT_DOWNLOAD_OK')


In [ ]:
import tarfile, os, glob, subprocess
DST = '/kaggle/working/frigid_pretrained_checkpoints.tar.gz'
with tarfile.open(DST) as t:
    members = t.getnames()
print('tar members:', len(members))
for m in members[:80]: print(m)
open('/kaggle/working/frigid_tar_list.txt', 'w').write('\n'.join(members))
OUT = '/kaggle/working/frigid_ckpts'
os.makedirs(OUT, exist_ok=True)
r = subprocess.run(['tar', '-xzf', DST, '-C', OUT], capture_output=True, text=True)
print('tar rc', r.returncode, (r.stderr or '')[-500:])
o = subprocess.run(['du', '-sh', OUT], capture_output=True, text=True); print(o.stdout)
files = [p for p in sorted(glob.glob(OUT + '/**/*', recursive=True)) if os.path.isfile(p)]
print('n_files', len(files))
for p in files: print(p, os.path.getsize(p))

In [ ]:
import subprocess, sys
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print('$', cmd, 'rc=', r.returncode)
    print((r.stdout or '')[-2000:])
    print((r.stderr or '')[-800:])
    return r
sh('git clone --depth 1 https://github.com/coleygroup/FRIGID /kaggle/working/FRIGID 2>&1 | tail -3')
sh('git -C /kaggle/working/FRIGID rev-parse HEAD')
sh('ls /kaggle/working/FRIGID; echo ---SRC---; ls /kaggle/working/FRIGID/src 2>/dev/null; echo ---CFG---; ls /kaggle/working/FRIGID/configs 2>/dev/null; echo ---SCRIPTS---; ls /kaggle/working/FRIGID/scripts 2>/dev/null')
sh('df -h /kaggle/working | tail -2')
# Deps required by FRIGID imports: dlm/sampler.py needs torch+numpy+safe(safe-mol)+rdkit+lightning(DLM module)+transformers;
# dlm/model.py is a Lightning module [src/dlm/model.py]; sampler header imports safe as sf, rdkit, dlm.model.DLM.
import numpy as _np0
print('numpy_before_pip', _np0.__version__)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'rdkit', 'selfies', 'safe-mol', 'omegaconf', 'lightning', 'transformers==4.56.2', 'einops', 'hydra-core', 'datasets', 'pandas', 'tqdm', 'pyyaml',
 'jaxtyping', 'pot', 'scikit-learn', 'matplotlib'], capture_output=True, text=True)
print('pip main rc', r.returncode)
rB = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'bionemo-moco'], capture_output=True, text=True)
print('pip bionemo-no-deps rc', rB.returncode)
print((r.stdout or '')[-800:], (r.stderr or '')[-1500:])
import importlib
for m in ('rdkit', 'selfies', 'safe', 'torch', 'omegaconf', 'lightning', 'transformers', 'einops', 'hydra', 'bionemo', 'datasets', 'pandas', 'tqdm', 'yaml', 'jaxtyping', 'pot', 'sklearn', 'matplotlib'):
    try:
        v = importlib.import_module(m); print(m, getattr(v, '__version__', 'ok'))
    except Exception as e: print(m, 'IMPORT_FAIL', repr(e)[:200])
import numpy as _np1
print('numpy_after_all_pip', _np1.__version__)
assert _np1.__version__.startswith('2.'), 'NUMPY_DOWNGRADED_ON_DISK'
import importlib as _il2
for m in ('numpy', 'numpy.random', 'safe', 'lightning', 'datasets', 'pandas', 'torch', 'rdkit', 'bionemo', 'hydra', 'jaxtyping', 'pot', 'sklearn', 'matplotlib'):
    try:
        _il2.import_module(m); print(m, 'RECHECK_OK')
    except Exception as e: print(m, 'RECHECK_FAIL', repr(e)[:200])
# API discovery (log evidence, no fabrication): list real entry points in the cloned tree.
sys.path.insert(0, '/kaggle/working/FRIGID')
sys.path.insert(0, '/kaggle/working/FRIGID/src')
import re as _re
for rel in ('src/dlm/sampler.py', 'src/dlm/utils/benchmark_utils.py', 'src/mist/models/spectra_encoder.py'):
    p = '/kaggle/working/FRIGID/' + rel
    try:
        src = open(p).read()
        print('---', rel, _re.findall(r'def (\w+)', src)[:30])
    except Exception as e:
        print(rel, 'READ_FAIL', repr(e)[:200])
try:
    import inspect
    from dlm.sampler import Sampler as _S
    print('Sampler.__init__:', inspect.signature(_S.__init__))
    print('unified_conditioned_generation:', inspect.signature(_S.unified_conditioned_generation))
    from dlm.utils import benchmark_utils as _bu
    print('normalize_formula probe:', _bu.normalize_formula('C27H33NO4'))
    print('inchikey1 probe:', _bu.get_inchikey_first_block('AAAA-BBBB-CCCC'))
except Exception as e:
    print('IMPORT_DISCOVERY_FAIL (cell5 retries with logging):', repr(e)[:500])

In [ ]:
import json, glob, os, sys, time, traceback, random
import numpy as np
SEED = 20261004
random.seed(SEED); np.random.seed(SEED)
print('=== FRIGID smoke: 1 molecule, oracle formula, B=10 ===')
print('API sources (coleygroup/FRIGID Apache-2.0, paper arXiv:2604.16648):')
print(' [1] dlm.sampler.Sampler(path, config_overrides) :: src/dlm/sampler.py class Sampler.__init__')
print(' [2] Sampler.unified_conditioned_generation(formula, fingerprint, num_samples, softmax_temp, randomness, target_lengths, min_add_len) :: src/dlm/sampler.py')
print(' [3] MIST ckpt: torch.load + strip "encoder." prefix + load_state_dict(strict=False) :: scripts/benchmark_spec2mol.py::load_mist_encoder')
print(' [4] normalize_formula / get_inchikey_first_block :: src/dlm/utils/benchmark_utils.py')
print(' [5] MIST SpectraEncoderGrowing defaults hidden 512 / magma 512 / out 4096 :: configs/spec2mol_benchmark_canopus.yaml')
print(' [6] gen defaults batch 16 / temp 1.0 / randomness 10.0 :: configs/spec2mol_benchmark_canopus.yaml [generation]')
print(' [7] predicted-formula: pilot_20.json mist_top1_formula <- artifacts/005/mist_cf_formula_topk_20.json (E4 15/20 top1)')
BANNED = ('fpnet', 'dreamsfp', 'dreams_fp', 'dreamms')
bad = [m for m in sys.modules if any(b in m.lower() for b in BANNED)]
print('contract guard (MIST-4096+formula only):', ('VIOLATION ' + str(bad)) if bad else 'OK none of FPNet-6930/DreamsFP/DreaMS-1024 imported')
try:
    import torch
    torch.manual_seed(SEED)
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
except Exception as e:
    print('TORCH_FAIL', repr(e)[:300])

def find_pilot():
    import glob as _g, os as _o
    cands = ['/kaggle/input/casmi26-frigid-pilot-20/pilot_20.json'] + sorted(_g.glob('/kaggle/input/**/pilot_20.json', recursive=True))
    for p in cands:
        if _o.path.isfile(p):
            print('pilot_20.json ->', p)
            return p
    raise FileNotFoundError('pilot_20.json absent; /kaggle/input=' + str(sorted(_o.listdir('/kaggle/input'))))
sys.path.insert(0, '/kaggle/working/FRIGID')
sys.path.insert(0, '/kaggle/working/FRIGID/src')

def discover_ckpts(root='/kaggle/working/frigid_ckpts'):
    files = [p for p in sorted(glob.glob(root + '/**/*', recursive=True)) if os.path.isfile(p)]
    info = [{'path': p, 'size': os.path.getsize(p)} for p in files]
    for d in info: print(f"  {d['size']:>12}  {d['path']}")
    print('n_ckpt_files', len(info))
    return info

def torch_probe(path):
    try:
        import torch
        obj = torch.load(path, map_location='cpu', weights_only=False)
        if isinstance(obj, dict):
            sd = obj.get('state_dict', obj)
            return {'top_keys': list(obj.keys())[:12], 'has_hparams': 'hyper_parameters' in obj,
                    'has_state_dict': 'state_dict' in obj, 'n_state': len(sd) if isinstance(sd, dict) else -1}
        return {'pytype': str(type(obj))}
    except Exception as e:
        return {'probe_error': repr(e)[:400]}

def pick_dlm_ckpt(info):
    scored = []
    for d in info:
        bn = os.path.basename(d['path']).lower()
        pr = torch_probe(d['path']); d['probe'] = pr
        s = 0
        if any(k in bn for k in ('dlm', 'frigid', 'decoder', 'fp2mol', 'base')): s += 3
        if 'mist' in bn or 'encod' in bn: s -= 5
        if pr.get('has_hparams') or pr.get('has_state_dict'): s += 2
        if d['size'] > 50_000_000: s += 1
        scored.append((s, d))
        print(f'  score={s} size={d["size"]} {d["path"]} probe={str(pr)[:220]}', flush=True)
    scored.sort(key=lambda t: (-t[0], -t[1]['size']))
    return scored[0][1] if scored else None

def load_sampler(dlm_path):
    # torch>=2.6 defaults torch.load weights_only=True; FRIGID DLM.ckpt hyper_parameters embed
    # OmegaConf objects -> lightning DLM.load_from_checkpoint [src/dlm/sampler.py::_load_checkpoint_with_fallback]
    # refuses each unallowlisted GLOBAL (v6/v7 worker logs). Register seed set, then adaptively
    # allowlist every further GLOBAL named by the error (ckpt is md5-verified official Zenodo 19685145).
    import re as _re2
    import torch as _tsg
    from dlm.sampler import Sampler
    notes = {'safe_globals_extra': []}
    try:
        from omegaconf import DictConfig as _DC, ListConfig as _LC
        _tsg.serialization.add_safe_globals([_DC, _LC])
        notes['safe_globals_seed'] = ['omegaconf.DictConfig', 'omegaconf.ListConfig']
    except Exception as e:
        notes['safe_globals_seed_fail'] = repr(e)[:200]

    # v9/v10 worker logs: weights-only unpickler structurally rejects defaultdict
    # ('Can only SETITEM ...') anywhere in the file, even in lightning-ignored sections;
    # allowlisting cannot fix that class. The ckpt is the md5-verified official Zenodo 19685145
    # artifact loaded in an isolated DEV kernel, so force weights_only=False for loads done
    # inside lightning DLM.load_from_checkpoint [src/dlm/sampler.py::_load_checkpoint_with_fallback]
    # via a contained monkeypatch (restored afterwards).
    import torch as _tl
    _real_torch_load = _tl.load

    def _trusted_load(*a, **k):
        k['weights_only'] = False
        return _real_torch_load(*a, **k)

    notes['dlm_load_path'] = dlm_path
    notes['weights_only'] = 'forced-False-via-contained-monkeypatch(trusted md5-verified ckpt)'

    def _make(mode):
        _tl.load = _trusted_load
        try:
            if mode == 'default':
                return Sampler(dlm_path)
            return Sampler(dlm_path, config_overrides={'use_shared_cross_attention': True})
        finally:
            _tl.load = _real_torch_load

    s = None
    for mode in ('default', 'shared_cross_attention'):
        for attempt in range(9):
            try:
                s = _make(mode)
                notes['mode'] = mode if attempt == 0 else f'{mode}+globals_retry{attempt}'
                break
            except Exception as e:
                m = _re2.search(r'Unsupported global: GLOBAL ([\w\.]+)', str(e))
                if m and attempt < 8:
                    dotted = m.group(1)
                    if '.' in dotted:
                        mod, name = dotted.rsplit('.', 1)
                        cls = getattr(__import__(mod, fromlist=[name]), name)
                    else:
                        import builtins as _bi
                        cls = getattr(_bi, dotted)
                    _tsg.serialization.add_safe_globals([cls])
                    notes['safe_globals_extra'].append(dotted)
                    print(f'safe_globals +{dotted}, retry {attempt + 1} [{mode}]', flush=True)
                    continue
                if mode == 'default':
                    notes['default_fail'] = repr(e)[:500]
                    print('Sampler default load failed, retry shared-cross-attention [scripts/benchmark_spec2mol.py::load_dlm_sampler]:', repr(e)[:300], flush=True)
                    break
                raise
        if s is not None:
            break
    assert s is not None, 'both sampler modes failed'
    print('fingerprint_conditioning:', s.model.use_fingerprint_conditioning, '| formula_conditioning:', s.model.use_formula_conditioning, '| fp_bits:', getattr(s, 'fingerprint_bits', '?'))
    notes.update({'fp_cond': bool(s.model.use_fingerprint_conditioning), 'formula_cond': bool(s.model.use_formula_conditioning)})
    return s, notes

def get_chem_helpers():
    try:
        from dlm.utils.benchmark_utils import normalize_formula as nf, get_inchikey_first_block as ik
        return nf, ik, 'repo:dlm.utils.benchmark_utils'
    except Exception as e:
        print('chem helper local fallback (Hill-notation port):', repr(e)[:200])
        import re as _re
        _pat = _re.compile(r'([A-Z][a-z]?)(\d*)')
        def nf(f):
            m = _pat.findall(str(f))
            if not m: return f
            c = {}
            for el, n in m:
                if el: c[el] = c.get(el, 0) + (int(n) if n else 1)
            oe = (['C'] if 'C' in c else []) + (['H'] if 'C' in c and 'H' in c else [])
            oe += sorted(x for x in c if x not in oe)
            return ''.join(f'{el}{c[el] if c[el] != 1 else ""}' for el in oe)
        def ik(k):
            return k.strip().upper().split('-')[0] if k else None
        return nf, ik, 'local-fallback(Hill port)'

def load_mist_status(info):
    # Weights-load check only. Forward needs mist-data Spec featurizer (subformula assignment)
    # which is not adapted for Enveda peaks here -> generation uses fingerprint=None (formula-only,
    # explicitly supported by unified_conditioned_generation docstring) and records fp_source honestly.
    st = {}
    cands = [d for d in info if 'mist' in os.path.basename(d['path']).lower() or 'encod' in os.path.basename(d['path']).lower()]
    if not cands:
        st['status'] = 'no-mist-ckpt-in-tarball'; print('MIST status:', st['status']); return None, st
    # push2 log: ckpt keys are 'spectra_encoder.0...' (pure encoder state, no 'encoder.' prefix).
    # try canopus-small (hidden 512/magma 512, configs/spec2mol_benchmark_canopus.yaml) then msg (640/2048), keep best.
    cfgs = [('canopus-small', dict(hidden_size=512, magma_modulo=512)), ('msg', dict(hidden_size=640, magma_modulo=2048))]
    base = dict(form_embedder='pos-cos', output_size=4096, spectra_dropout=0.1, peak_attn_layers=2, num_heads=8, set_pooling='cls', refine_layers=4, pairwise_featurization=True, embed_instrument=False, inten_transform='float', inten_prob=0.1, remove_prob=0.5, cls_type='ms1', spec_features='peakformula', mol_features='fingerprint', top_layers=1)
    best = None
    try:
        import torch
        from mist.models import SpectraEncoderGrowing
        for p in sorted([d['path'] for d in cands], key=lambda q: (0 if 'canopus' in q else 1)):
            ck = torch.load(p, map_location='cpu', weights_only=False)
            sd = ck.get('state_dict', ck)
            sd = {k.replace('encoder.', ''): v for k, v in sd.items() if k.startswith('encoder.')} or sd
            for cname, co in cfgs:
                try:
                    enc = SpectraEncoderGrowing(**dict(base, **co))
                    miss, unexp = enc.load_state_dict(sd, strict=False)
                    cand = (len(miss), len(unexp), p, cname, enc)
                    print(f'MIST try {p.split("/")[-1]} [{cname}]: missing={len(miss)} unexpected={len(unexp)}')
                    if best is None or (len(miss), len(unexp)) < (best[0], best[1]):
                        best = cand
                except Exception as e:
                    print(f'MIST try {cname} build/load fail:', repr(e)[:200])
        if best:
            best[4].eval()
            st.update({'status': 'weights-loaded-forward-pending', 'path': best[2], 'cfg': best[3], 'missing': best[0], 'unexpected': best[1]})
            print(f"MIST weights loaded [{best[3]}]; forward pending Spec-featurizer adaptation")
            return best[4], st
        st.update({'status': 'load-failed', 'error': 'all configs failed'})
        return None, st
    except Exception as e:
        st.update({'status': 'load-failed', 'error': repr(e)[:500]})
        print('MIST load failed:', repr(e)[:500])
        return None, st

def run_arm(sampler, nf, ik1, formula_raw, truth_smiles, B, batch_size=10, softmax_temp=1.0, randomness=10.0):
    from rdkit import Chem
    from rdkit.Chem import rdMolDescriptors
    target = nf(formula_raw)
    truth_can = None
    try:
        _m = Chem.MolFromSmiles(truth_smiles)
        truth_can = Chem.MolToSmiles(_m, canonical=True) if _m else None
    except Exception:
        pass
    uniq = {}
    n_valid = 0; n_gen = 0; order = 0
    n_none_empty = 0; n_unparseable = 0; bad_examples = []
    t0 = time.time(); gen_sec = 0.0
    while n_gen < B:
        cur = min(batch_size, B - n_gen)
        tg = time.time()
        samples = sampler.unified_conditioned_generation(formula=target, fingerprint=None, num_samples=cur, softmax_temp=softmax_temp, randomness=randomness, min_add_len=2)
        gen_sec += time.time() - tg
        if not samples: samples = [None] * cur
        n_gen += len(samples)
        for s in samples:
            if not s:
                n_none_empty += 1
                continue
            try:
                m = Chem.MolFromSmiles(str(s))
                if m is None:
                    n_unparseable += 1
                    if len(bad_examples) < 5: bad_examples.append(str(s)[:300])
                    continue
                n_valid += 1
                canon = Chem.MolToSmiles(m, canonical=True)
                try: key = ik1(Chem.MolToInchiKey(m))
                except Exception: key = 'NOINCHI:' + canon
                if key in uniq or key is None: continue
                try: gf = nf(rdMolDescriptors.CalcMolFormula(m))
                except Exception: gf = None
                uniq[key] = {'smiles': canon, 'formula_match': (gf == target), 'order': order}; order += 1
            except Exception:
                continue
        print(f'  progress {n_gen}/{B} valid={n_valid} uniq={len(uniq)}', flush=True)
    seq = sorted(uniq.values(), key=lambda d: d['order'])
    matched = [v['smiles'] for v in seq if v['formula_match']]
    rest = [v['smiles'] for v in seq if not v['formula_match']]
    have = set(v['smiles'] for v in seq)
    novel = sum(1 for v in seq if v['smiles'] != truth_can)
    return {'formula_raw': formula_raw, 'formula_norm': target, 'B': B, 'n_generated': n_gen,
            'n_valid': n_valid, 'valid_rate': n_valid / max(n_gen, 1), 'n_unique': len(uniq),
            'unique_rate': len(uniq) / max(n_gen, 1), 'n_novel': novel,
            'novelty_rate': novel / max(len(uniq), 1), 'n_formula_matched': len(matched),
            'truth_in_pool': bool(truth_can and truth_can in have), 'ranked25': (matched + rest)[:25],
            'n_none_empty': n_none_empty, 'n_unparseable': n_unparseable, 'bad_examples': bad_examples,
            'rank_method': 'formula-matched-first+generation-order (fingerprint=None; no MIST pred-FP on worker)',
            'gen_seconds': round(gen_sec, 1), 'total_seconds': round(time.time() - t0, 1)}

res = {'seed': SEED, 'smoke': 'NOT_RUN'}
t0 = time.time()
try:
    info = discover_ckpts()
    assert info, 'NO_FILES_AFTER_UNPACK'
    dlm = pick_dlm_ckpt(info)
    assert dlm, 'NO_DLM_CKPT_FOUND'
    res['dlm_ckpt'] = {'path': dlm['path'], 'size': dlm['size'], 'probe': dlm.get('probe')}
    sampler, snotes = load_sampler(dlm['path'])
    res['sampler'] = snotes
    NF, IK1, chem_src = get_chem_helpers()
    res['chem_helpers'] = chem_src
    _, mist_st = load_mist_status(info)
    res['mist_encoder'] = mist_st
    try:
        res['model_cfg'] = {'use_bracket_safe': sampler.model.config.training.get('use_bracket_safe'),
                            'use_fp_cond': bool(sampler.model.use_fingerprint_conditioning),
                            'use_formula_cond': bool(sampler.model.use_formula_conditioning),
                            'fp_bits': getattr(sampler, 'fingerprint_bits', None)}
        print('model_cfg:', json.dumps(res['model_cfg']))
    except Exception as e:
        res['model_cfg'] = {'err': repr(e)[:200]}
    # SAFE roundtrip (isolates tokenizer+safe stack, no model): SMILES->SAFE->tokens->SAFE->SMILES
    try:
        import safe as _sf
        from dlm.utils.utils_chem import safe_to_smiles as _s2s
        _m = __import__('rdkit.Chem', fromlist=['MolFromSmiles']).MolFromSmiles('CCO')
        _enc = _sf.SAFEConverter(slicer=None).encoder('CCO')
        _tok = sampler.model.tokenizer([_enc], return_tensors='pt', truncation=True,
                                       max_length=sampler.model.config.model.max_position_embeddings)['input_ids']
        _dec = sampler.model.tokenizer.batch_decode(_tok, skip_special_tokens=True)
        _back = _s2s(_dec[0], fix=True) if _dec else None
        res['safe_roundtrip'] = {'safe': str(_enc)[:200], 'decoded': str(_dec[0])[:200] if _dec else None, 'back': _back}
        print('SAFE_ROUNDTRIP:', json.dumps(res['safe_roundtrip'])[:500])
    except Exception as e:
        res['safe_roundtrip'] = {'err': traceback.format_exc()[-800:]}
        print('SAFE_ROUNDTRIP_FAIL:', repr(e)[:300])
    # de-novo probe (no conditioning): if valid>0, decoder+tokenizer work; issue is conditioning path
    try:
        _dn = sampler.de_novo_generation(num_samples=5, softmax_temp=1.0, randomness=10.0)
        from rdkit import Chem as _RC
        _dnv = sum(1 for s in (_dn or []) if s and _RC.MolFromSmiles(str(s)))
        res['denovo_probe'] = {'n': len(_dn or []), 'n_valid': _dnv,
                               'samples': [str(s)[:200] for s in (_dn or [])[:3]]}
        print('DENOVO_PROBE:', json.dumps(res['denovo_probe'])[:800])
    except Exception as e:
        res['denovo_probe'] = {'err': traceback.format_exc()[-800:]}
        print('DENOVO_PROBE_FAIL:', repr(e)[:300])
    res['fp_source'] = 'fingerprint=None formula-only (MIST encoder forward pending Spec-featurizer adaptation)'
    pilot = json.load(open(find_pilot()))
    print('pilot n =', pilot['meta']['n'])
    m0 = pilot['molecules'][0]
    print('mol0', m0['key'], 'truth=', m0['truth_formula'], 'mist_top1=', m0['mist_top1_formula'])
    arm = run_arm(sampler, NF, IK1, m0['truth_formula'], m0['truth_smiles'], B=10, batch_size=10)
    res['mol'] = m0['key']
    res['arm_oracle'] = arm
    res['smoke'] = 'GEN_OK' if arm['n_valid'] >= 1 else 'SMOKE_NO_VALID'
    print('SMOKE', res['smoke'], 'valid=%d/10 uniq=%d matched=%d truth_in_pool=%s' % (arm['n_valid'], arm['n_unique'], arm['n_formula_matched'], arm['truth_in_pool']))
except Exception:
    res['smoke'] = 'SMOKE_BLOCKED'
    res['traceback'] = traceback.format_exc()[-3000:]
    print(res['traceback'])
res['seconds'] = round(time.time() - t0, 1)
json.dump(res, open('/kaggle/working/frigid_smoke.json', 'w'), indent=1)
print(json.dumps({k: (v if k in ('smoke', 'mol', 'seconds', 'fp_source', 'chem_helpers') else '...') for k, v in res.items()}, indent=1))


In [ ]:
import json, time, traceback
smoke = json.load(open('/kaggle/working/frigid_smoke.json'))
print('smoke status:', smoke.get('smoke'))
def find_pilot():
    import glob as _g, os as _o
    cands = ['/kaggle/input/casmi26-frigid-pilot-20/pilot_20.json'] + sorted(_g.glob('/kaggle/input/**/pilot_20.json', recursive=True))
    for p in cands:
        if _o.path.isfile(p):
            print('pilot_20.json ->', p)
            return p
    raise FileNotFoundError('pilot_20.json absent; /kaggle/input=' + str(sorted(_o.listdir('/kaggle/input'))))

pilot = json.load(open(find_pilot()))
t0 = time.time()
out = {'meta': {'smoke': smoke.get('smoke'), 'n_target': 20, 'n_attempted': 0, 'arms': ['oracle-formula', 'mistcf-top1'],
        'iceberg': 'OFF_round0', 'seed': 20261004, 'B_per_arm': 100,
        'fingerprint': 'MIST-4096 only; FPNet-6930/DreamsFP/DreaMS-1024 never imported',
        'fp_source': smoke.get('fp_source'), 'formula_provenance': 'oracle=truth_formula, mistcf-top1=mist_top1_formula (E4 mist_cf_formula_topk_20.json 15/20)',
        'scoring': 'ported src/evaluation.py::reciprocal_rank_by_position (final-position, no dedup, empty dropped, >25 rejected)'}, 'per_molecule': []}

def rr_by_position(ranked, truth_smiles, k=25):
    # Port of src/evaluation.py::reciprocal_rank_by_position (_repo semantics, RDKit canonical match on worker).
    from rdkit import Chem
    cleaned = [s for s in (str(s).strip() for s in ranked) if s != '']
    if len(cleaned) == 0 or len(cleaned) > k:
        return {'rank': None, 'rr': 0.0, 'caliber': 'rejected', 'n': len(cleaned)}
    try:
        _tm = Chem.MolFromSmiles(truth_smiles)
        tk = Chem.MolToSmiles(_tm, canonical=True) if _tm else truth_smiles
        cal = 'rdkit-canonical(port of src/evaluation.py)'
    except Exception:
        tk, cal = truth_smiles, 'exact-string-fallback'
    for i, p in enumerate(cleaned):
        try:
            _pm = Chem.MolFromSmiles(p)
            pk = Chem.MolToSmiles(_pm, canonical=True) if _pm else None
        except Exception:
            pk = None
        if pk is not None and pk == tk:
            return {'rank': i + 1, 'rr': 1.0 / (i + 1), 'caliber': cal, 'n': len(cleaned)}
    return {'rank': None, 'rr': 0.0, 'caliber': cal, 'n': len(cleaned)}

if smoke.get('smoke') != 'GEN_OK':
    out['meta']['status'] = 'PILOT_BLOCKED_ON_SMOKE'
    out['meta']['smoke_traceback'] = smoke.get('traceback', '')[:2000]
    print('PILOT_BLOCKED: 1-molecule smoke harness not green; not fabricating 20-mol numbers.')
    print('卡点:', (smoke.get('traceback', '') or smoke.get('smoke', ''))[-1500:])
else:
    try:
        BANNED = ('fpnet', 'dreamsfp', 'dreams_fp', 'dreamms')
        import sys as _sys
        _bad = [m for m in _sys.modules if any(b in m.lower() for b in BANNED)]
        assert not _bad, f'CONTRACT_VIOLATION {_bad}'
        agree = sum(1 for m in pilot['molecules'] if m['mist_top1_formula'] == m['truth_formula'])
        print(f'formula agreement mist_top1==truth: {agree}/{len(pilot["molecules"])} (E4 offline: 15/20)')
        rows = []
        for mi, m in enumerate(pilot['molecules']):
            t1 = time.time()
            arms = {}
            for aname, fml in (('oracle', m['truth_formula']), ('mistcf-top1', m['mist_top1_formula'])):
                a = run_arm(sampler, NF, IK1, fml, m['truth_smiles'], B=100, batch_size=16)
                r_main = rr_by_position(a['ranked25'], m['truth_smiles'])
                # truth-excluded diagnostic (B-track library-out view): drop truth canonical explicitly
                from rdkit import Chem as _C
                try:
                    _tc = _C.MolToSmiles(_C.MolFromSmiles(m['truth_smiles']), canonical=True)
                except Exception:
                    _tc = m['truth_smiles']
                r_ex = rr_by_position([s for s in a['ranked25'] if s != _tc], m['truth_smiles'])
                a.update({'rr_with_truth': r_main, 'rr_truth_excluded': r_ex})
                arms[aname] = a
                print(f"[{mi+1}/20] {m['key']} {aname}: valid={a['n_valid']}/100 uniq={a['n_unique']} matched={a['n_formula_matched']} recall={a['truth_in_pool']} rr={r_main['rr']:.3f} dt={a['total_seconds']}s", flush=True)
            rows.append({'key': m['key'], 'truth_formula': m['truth_formula'], 'mist_top1': m['mist_top1_formula'],
                         'oracle': arms['oracle'], 'mistcf-top1': arms['mistcf-top1'],
                         'seconds': round(time.time() - t1, 1)})
            out['meta']['n_attempted'] = mi + 1
        def agg(rows, an):
            vv = [r[an] for r in rows]
            return {'n': len(vv), 'mean_valid_rate': sum(v['valid_rate'] for v in vv) / len(vv),
                    'mean_unique_rate': sum(v['unique_rate'] for v in vv) / len(vv),
                    'mean_novelty_rate': sum(v['novelty_rate'] for v in vv) / len(vv),
                    'pool_recall_truth': sum(1 for v in vv if v['truth_in_pool']) / len(vv),
                    'mrr_with_truth': sum(v['rr_with_truth']['rr'] for v in vv) / len(vv),
                    'mrr_truth_excluded': sum(v['rr_truth_excluded']['rr'] for v in vv) / len(vv)}
        out['meta']['status'] = 'COMPLETE_20x2'
        out['meta']['agg_oracle'] = agg(rows, 'oracle')
        out['meta']['agg_mistcf-top1'] = agg(rows, 'mistcf-top1')
        out['per_molecule'] = rows
        print('AGG_ORACLE', json.dumps(out['meta']['agg_oracle'], indent=1))
        print('AGG_MISTCF-TOP1', json.dumps(out['meta']['agg_mistcf-top1'], indent=1))
    except Exception:
        out['meta']['status'] = 'PILOT_FAILED_MIDRUN'
        out['meta']['traceback'] = traceback.format_exc()[-3000:]
        print(out['meta']['traceback'])
json.dump(out, open('/kaggle/working/frigid_pilot_results.json', 'w'), indent=1)
print('elapsed', round(time.time() - t0, 1), 's; wrote /kaggle/working/frigid_pilot_results.json')
print(json.dumps(out['meta'], indent=1, ensure_ascii=False)[:3000])
